# 🧠 การวิเคราะห์ขนาดของ Batch: สัญญาณรบกวน, หน่วยความจำ และกฎการปรับขนาด

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **Batch Size Analysis**! ในสมุดบันทึกนี้ เราจะ:
1. กำหนดรูปแบบความสัมพันธ์ระหว่างขนาดของ batch และสัญญาณรบกวนของเกรเดียนต์ (Central Limit Theorem)
2. สร้างข้อมูลจำลองและคำนวณค่าความแปรปรวนเชิงตัวเลขของการประมาณการเกรเดียนต์ในขนาด batch ต่างๆ
3. พล็อตกราฟ **Gradient Noise vs. Batch Size** เพื่อแสดงภาพกฎการสลายตัวของสัญญาณรบกวนแบบ $1/\sqrt{B}$
4. นำกฎการปรับขนาดเชิงเส้น (**Linear Scaling Rule**) ไปใช้เพื่อปรับอัตราการเรียนรู้ (learning rates) เมื่อขยายขนาด batch
5. ประมาณการข้อจำกัดของการขยายขนาดหน่วยความจำ VRAM
6. เชื่อมโยงหลักการเหล่านี้กับการตั้งค่า batch ของ YOLO และการแก้ไขปัญหา CUDA Out of Memory (OOM)

มาเริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันก่อน

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. การวัดปริมาณสัญญาณรบกวนของเกรเดียนต์เชิงตัวเลข

ตามทฤษฎีขีดจำกัดส่วนกลาง (Central Limit Theorem) ค่าความแปรปรวนของเกรเดียนต์เฉลี่ยตัวอย่างจะลดลงตามสัดส่วน $1/B$ (โดยที่ $B$ คือขนาดของ batch) ดังนััน ค่าเบี่ยงเบนมาตรฐาน (สัญญาณรบกวน) จะสลายตัวตามสูตร:
$$\text{Noise} \propto \frac{1}{\sqrt{B}}$$

มาสร้างจุดข้อมูล 200 จุด และประเมินความสัมพันธ์นี้กัน

In [ ]:
n_samples = 200
X = np.random.rand(n_samples, 1)
y = 2.0 * X + 1.0 + np.random.normal(0, 0.2, (n_samples, 1))

# We evaluate the gradient at a fixed weight w=1.5 and bias b=0.5
w_test, b_test = 1.5, 0.5

def single_sample_gradient(xi, yi, w, b):
    pred = w * xi + b
    return (pred - yi) * xi

true_grads = [single_sample_gradient(X[i], y[i], w_test, b_test) for i in range(n_samples)]
true_mean_grad = np.mean(true_grads)

batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128]
std_deviations = []

for B in batch_sizes:
    batch_gradients = []
    for _ in range(150):
        indices = np.random.choice(n_samples, size=B, replace=False)
        sample_grads = [single_sample_gradient(X[i], y[i], w_test, b_test) for i in indices]
        batch_gradients.append(np.mean(sample_grads))
    
    std_deviations.append(np.std(batch_gradients))

มาพล็อตค่าเบี่ยงเบนมาตรฐานของการประมาณการเกรเดียนต์เปรียบเทียบกับขนาด batch โดยซ้อนทับเส้นโค้งการสลายตัวตามทฤษฎีแบบ $1/\sqrt{B}$ กัน

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(batch_sizes, std_deviations, color='red', marker='o', linewidth=2.5, label='Empirical Gradient Noise (Std Dev)')

theoretical = std_deviations[0] / np.sqrt(batch_sizes)
plt.plot(batch_sizes, theoretical, color='blue', linestyle='--', linewidth=2, label='Theoretical 1/\sqrt{B} Decay')

plt.xscale('log', base=2)
plt.xlabel('Batch Size (B)')
plt.ylabel('Gradient Estimate Noise (Std Dev)')
plt.title('Gradient Noise Reduction vs. Batch Size')
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.legend()
plt.show()

ดูที่พล็อตสิ!
-   **Batch Size = 1 (Stochastic GD):** มีสัญญาณรบกวนสูงมาก การอัปเดตเกรเดียนต์สามารถชี้ไปในทิศทางที่ผิดพลาดอย่างมากในแต่ละขั้นตอน แต่สิ่งนี้จะช่วยเพิ่ม regularization ที่เป็นประโยชน์
-   **Batch Size = 32 หรือ 64:** สัญญาณรบกวนลดลงไปประมาณ 5-8 เท่า ขั้นตอนมีความเสถียรและเชื่อถือได้สูง
-   ข้อมูลจากการทดลองตรงกับเส้นโค้งการสลายตัวตามทฤษฎี $1/\sqrt{B}$ เกือบจะสมบูรณ์แบบ!

## 2. กฎการปรับขนาดเชิงเส้น (Linear Scaling Rule)

เมื่อปรับขนาด batch จากการกำหนดค่าเริ่มต้น เราจะปรับอัตราการเรียนรู้ตามสัดส่วนดังนี้:
$$\text{lr}_{\text{scaled}} = \text{lr}_{\text{base}} \times \frac{\text{batch}_{\text{target}}}{\text{batch}_{\text{base}}}$$

In [ ]:
def scale_learning_rate(lr_base, batch_base, batch_target):
    scale_factor = batch_target / batch_base
    return lr_base * scale_factor

print("Scaled LR at batch=64:", scale_learning_rate(0.01, 16, 64))

## 💡 การเชื่อมโยงไปยัง YOLO และ CUDA OOM
*   **ปริมาณการใช้ VRAM:** ในบันทึกการกำหนดค่าของ YOLO คุณสามารถดูการตั้งค่าขนาด batch ได้ (เช่น `batch=16`) ความต้องการใช้หน่วยความจำจะเพิ่มขึ้นเป็นเส้นตรงตามขนาดของ batch แต่เพิ่มขึ้นเป็น **กำลังสอง** ตามขนาดของรูปภาพ:
    $$\text{VRAM} \propto \text{Batch} \times \text{ImageHeight} \times \text{ImageWidth}$$
*   **CUDA OOM:** หาก GPU ของคุณแสดงข้อผิดพลาด CUDA Out of Memory:
    1.  หารขนาด `batch` ด้วย 2 (เช่น จาก 16 เป็น 8)
    2.  หากยังไม่เพียงพอ ให้ลดขนาดของรูปภาพ `imgsz` (เช่น จาก 640 เป็น 480)
    3.  หากต้องการรักษาขนาด batch ที่มีผลจริงเป็น 16 แต่มี VRAM จำกัดสำหรับขนาด batch เพียง 8 ให้ทำการอัปเดตเกรเดียนต์ทุกๆ 2 ขั้นตอน (gradient accumulation)